In [ ]:
import pandas as pd

# 👉 Replace with your actual path
file_path = "/path/to/your/data/Original_dice_all_fold.xlsx"

df = pd.read_excel(file_path) # Changed to pd.read_excel

print("Preview of dataset:")
print(df.head())

In [ ]:
#case-level failure
threshold = 0.70

# Select cases where ANY class has Dice < 0.70
failure_cases = df[df[class_cols].lt(threshold).any(axis=1)].copy()

print("Number of failure cases:", len(failure_cases))

In [ ]:
#Structure-level failure
def get_failed_classes(row):
    return [col for col in class_cols if row[col] < threshold]

failure_cases["Failed_Classes"] = failure_cases.apply(get_failed_classes, axis=1)

print(failure_cases[["Case", "Failed_Classes"]].head())

In [ ]:
thresholds = [0.60, 0.70, 0.80]

for threshold in thresholds:
  
    current_failure_df = df[df[class_cols].lt(threshold).any(axis=1)].copy()

    failure_cases = (
        current_failure_df["Case"]
        .drop_duplicates()
        .sort_values()
    )

    print(f"\nDSC < {threshold}")
    print("Number of failure cases:", len(failure_cases))
    .
    print("Failure rate:", len(failure_cases) / df["Case"].nunique() * 100)

In [ ]:
Mean and Percentile evaluation of Dice Score

In [ ]:
import numpy as np


dice = df[class_cols].stack().dropna()

print("N:", len(dice))
print("Mean:", dice.mean())
print("SD:", dice.std())
print("Median:", dice.median())
print("Minimum:", dice.min())
print("10th percentile:", np.percentile(dice, 10))
print("25th percentile:", np.percentile(dice, 25))
print("75th percentile:", np.percentile(dice, 75))
print("90th percentile:", np.percentile(dice, 90))
print("Maximum:", dice.max())

In [ ]:
quantiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

for q in quantiles:
    print(f"{q}th percentile:",
          np.percentile(dice, q))

In [ ]:
import numpy as np

# --------------------------------------------------
# Vertebrae: Class 1-5
# IVDs: Class 6-11
# --------------------------------------------------

vertebra_cols = [f'Class_{i}' for i in range(1, 6)]
ivd_cols = [f'Class_{i}' for i in range(6, 12)]


# --------------------------------------------------
# Threshold-free distribution statistics
# --------------------------------------------------

def distribution_stats(df, columns, name):

    dice = df[columns].stack().dropna()

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("N:", len(dice))
    print("Mean:", dice.mean())
    print("SD:", dice.std())
    print("Median:", dice.median())
    print("Minimum:", dice.min())
    print("10th percentile:", np.percentile(dice, 10))
    print("25th percentile:", np.percentile(dice, 25))
    print("75th percentile:", np.percentile(dice, 75))
    print("90th percentile:", np.percentile(dice, 90))
    print("Maximum:", dice.max())


# --------------------------------------------------
# Vertebrae distribution
# --------------------------------------------------

distribution_stats(
    df,
    vertebra_cols,
    "VERTEBRAE (Class 1-5)"
)


# --------------------------------------------------
# IVD distribution
# --------------------------------------------------

distribution_stats(
    df,
    ivd_cols,
    "IVDs (Class 6-11)"
)

In [ ]:
import numpy as np

# Class 1-5 = Vertebrae
vertebra_cols = [f'Class_{i}' for i in range(1, 6)]

# Class 6-11 = IVDs
ivd_cols = [f'Class_{i}' for i in range(6, 12)]


quantiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]


# -------------------------------
# Vertebrae
# -------------------------------

vertebra_dice = df[vertebra_cols].stack().dropna().values

print("=" * 50)
print("VERTEBRAE (Class 1-5)")
print("=" * 50)

for q in quantiles:
    print(f"{q}th percentile:",
          np.percentile(vertebra_dice, q))


# -------------------------------
# IVDs
# -------------------------------

ivd_dice = df[ivd_cols].stack().dropna().values

print("\n" + "=" * 50)
print("IVDs (Class 6-11)")
print("=" * 50)

for q in quantiles:
    print(f"{q}th percentile:",
          np.percentile(ivd_dice, q))

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import t

df = pd.read_excel("your_file.xlsx")

vertebra_cols = ["L5", "L4", "L3", "L2", "L1"]

# Patient-level mean
patient_mean = df[vertebra_cols].mean(axis=1)

n = patient_mean.notna().sum()
mean_dice = patient_mean.mean()
sd_patient = patient_mean.std(ddof=1)

se = sd_patient / np.sqrt(n)
tcrit = t.ppf(0.975, df=n-1)

ci_low = mean_dice - tcrit * se
ci_high = mean_dice + tcrit * se

print("N patients:", n)
print("Mean:", mean_dice)
print("95% CI:", ci_low, ci_high)

Patient-clustered 95% CI

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import t

#df = pd.read_excel("your_file.xlsx")

vertebra_cols = [f'Class_{i}' for i in range(1, 6)]

# Patient-level mean
patient_mean = df[vertebra_cols].mean(axis=1)

n = patient_mean.notna().sum()
mean_dice = patient_mean.mean()
sd_patient = patient_mean.std(ddof=1)

se = sd_patient / np.sqrt(n)
tcrit = t.ppf(0.975, df=n-1)

ci_low = mean_dice - tcrit * se
ci_high = mean_dice + tcrit * se

print("N patients:", n)
print("Mean:", mean_dice)
print("95% CI:", ci_low, ci_high)

In [ ]:

ivd_cols = [f'Class_{i}' for i in range(6, 12)]

patient_mean = df[ivd_cols].mean(axis=1)

n = patient_mean.notna().sum()
mean_dice = patient_mean.mean()
sd_patient = patient_mean.std(ddof=1)

se = sd_patient / np.sqrt(n)
tcrit = t.ppf(0.975, df=n-1)

ci_low = mean_dice - tcrit * se
ci_high = mean_dice + tcrit * se

print("N patients:", n)
print("Mean:", mean_dice)
print("95% CI:", ci_low, ci_high)

Structure-level analysis

In [ ]:
# Calculate mean, SD, and number of cases for each vertebral level

# First, transform the DataFrame to a long format for vertebrae
vertebra_df_long = df.melt(
    id_vars=['Case'],
    value_vars=vertebra_cols, # Use the already defined vertebra_cols (Class_1 to Class_5)
    var_name='class_name',
    value_name='dice'
)

# Extract the vertebra level from the class name (e.g., 'Class_1' -> '1')
vertebra_df_long['vertebra_level'] = vertebra_df_long['class_name'].str.extract('Class_(\d+)').astype(int)

vertebra_stats = (
    vertebra_df_long.groupby("vertebra_level")["dice"]
      .agg(["mean", "std", "count"])
      .reset_index()
)

# Create Mean ± SD column
vertebra_stats["Dice (Mean ± SD)"] = (
    vertebra_stats["mean"].round(3).astype(str)
    + " \u00B1 " # Unicode for plus-minus sign
    + vertebra_stats["std"].round(3).astype(str)
)

# Keep required columns
vertebra_stats = vertebra_stats[
    ["vertebra_level", "Dice (Mean \u00B1 SD)", "count"]
]

# Rename count
vertebra_stats = vertebra_stats.rename(
    columns={"count": "Number of Cases"}
)

print(vertebra_stats)

In [ ]:
# Calculate mean, SD, and number of cases for each IVD level

# First, transform the DataFrame to a long format for IVD
disc_df_long = df.melt(
    id_vars=['Case'],
    value_vars=ivd_cols, # Corrected: Use ivd_cols
    var_name='class_name',
    value_name='dice'
)

# Extract the vertebra level from the class name (e.g., 'Class_1' -> '1')
disc_df_long['intervertebral disc_level'] = disc_df_long['class_name'].str.extract('Class_(\d+)').astype(int)

disc_stats = (
    disc_df_long.groupby("intervertebral disc_level")["dice"]
      .agg(["mean", "std", "count"])
      .reset_index()
)

# Create Mean ± SD column
disc_stats["Dice (Mean ± SD)"] = (
    disc_stats["mean"].round(3).astype(str)
    + " \u00B1 " # Unicode for plus-minus sign
    + disc_stats["std"].round(3).astype(str)
)

# Keep required columns
disc_stats = disc_stats[
    ["intervertebral disc_level", "Dice (Mean \u00B1 SD)", "count"] # Corrected: Use 'intervertebral disc_level'
]

# Rename count
disc_stats = disc_stats.rename(
    columns={"count": "Number of Cases"}
)

print(disc_stats)